## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [7]:
## add your code here
import sys

class Perm:
    def __init__(self, arr, n):
        self.a = arr[:]
        self.n = n
        self.vec = []

    def calc(self):
        if self.n == 1:
            return True
            
        b_arr = [0] * (self.n // 2)
        c_arr = [0] * (self.n // 2)
        
        for i in range(self.n // 2):
            b_arr[i] = self.a[i * 2] // 2
            c_arr[i] = self.a[i * 2 + 1] // 2
            
        b = Perm(b_arr, self.n // 2)
        c = Perm(c_arr, self.n // 2)
        
        if not b.calc() or not c.calc():
            return False
            
        if self.a[0] % 2 != 0:
            self.vec.append(1 if self.n == 2 else -1)
            
        tb = 0
        for val in b.vec:
            if val > 0:
                self.vec.extend([-1, 1])
            else:
                self.vec.append(val * 2)
                tb ^= (-val * 2)
                
        if tb != 0:
            self.vec.append(-tb)
            
        tc = 0
        for val in c.vec:
            if val > 0:
                self.vec.extend([1, -1])
            else:
                self.vec.append(val * 2)
                tc ^= (-val * 2)
                
        if (tc & (self.n // 2)) != (tb & (self.n // 2)):
            return False
            
        if tb >= (self.n // 2):
            tb -= self.n // 2
        if tc >= (self.n // 2):
            tc -= self.n // 2
            
        if tb != tc:
            return False
            
        # 合并连续的异或操作
        tmp = []
        for val in self.vec:
            if not tmp:
                tmp.append(val)
            else:
                if val < 0 and tmp[-1] < 0:
                    tmp[-1] = -((-tmp[-1]) ^ (-val))
                    if tmp[-1] == 0:
                        tmp.pop()
                else:
                    tmp.append(val)
                    
        self.vec = tmp
        return True


class Solver:
    def __init__(self, n, a, b, o):
        self.n = n
        self.a = a
        self.b = b
        self.o = o
        self.ans = []
        self.l = 0

    # 阶段一：真实修改数组的全局操作
    def do_add(self, x):
        if x == 0: return
        self.ans.append(x)
        self.o = [(v + x) % self.n for v in self.o]

    def do_xor(self, x):
        if x == 0: return
        self.ans.append(-x)
        self.o = [v ^ x for v in self.o]
    # 仅记录操作，不修改全局数组
    def record_add(self, x):
        if x == 0: return
        self.ans.append(x)

    def record_xor(self, x):
        if x == 0: return
        self.ans.append(-x)

    def record_magic(self):
        self.ans.append(0)

    def calc_offset(self, start, end):
        """计算 start 到 end 的异或和加法偏移量"""
        delta = (end - start + self.n - self.l + self.n) % self.n
        pa = pb = 0
        stp = self.n // 2
        while stp >= 2 * self.l:
            if delta >= stp:
                delta -= stp
                pb += stp // 2
            else:
                pa += stp // 2
            stp //= 2
            
        pa += self.n // 2
        pa += (start & (self.l - 1))
        pb += (start & (self.l - 1))
        return pa, pb

    def do_swap_simulate(self, c, d):
        """仅记录操作，不更新数组"""
        if (c // self.l) % 2 == (d // self.l) % 2:
            if (c // self.l) % 2 == 0:
                p = (c & (self.l - 1)) + self.l
            else:
                p = (c & (self.l - 1))
            self.do_swap_simulate(c, p)
            self.do_swap_simulate(d, p)
            self.do_swap_simulate(c, p)
        else:
            pa, pb = self.calc_offset(self.a, self.b)
            pc, pd = self.calc_offset(c, d)

            self.record_add((pc - c + self.n) % self.n)
            self.record_xor(pc ^ pa)
            self.record_add((self.a - pa + self.n) % self.n)
            
            self.record_magic()
            
            self.record_add((pa - self.a + self.n) % self.n)
            self.record_xor(pc ^ pa)
            self.record_add((c - pc + self.n) % self.n)


def main():
    # 一次性读取所有输入
    input_data = sys.stdin.read().split()
    if not input_data:
        return
        
    n = int(input_data[0])
    a = int(input_data[1])
    b = int(input_data[2])
    o = [int(x) for x in input_data[3:3+n]]
    
    solver = Solver(n, a, b, o)
    
    l = (solver.a - solver.b + solver.n) % solver.n
    l = l & -l 
    if l == 0:
        l = solver.n
    solver.l = l
    
    # 阶段一：低位对齐，真实更新数组
    if l > 1:
        arr = [solver.o[i] & (l - 1) for i in range(solver.n)]
        perm = Perm(arr[:l], l)
        
        if not perm.calc():
            print("-1")
            return
            
        for val in perm.vec:
            if val > 0:
                solver.do_add(val)
            else:
                solver.do_xor(-val)
                
    # 阶段二：利用共轭变换局部模拟排序
    for i in range(l):
        vec = []
        for j in range(i, solver.n, l):
            vec.append(solver.o[j])
            
        vec.sort()
        flag = True
        c = 0
        for j in range(i, solver.n, l):
            if vec[c] != j:
                flag = False
                break
            c += 1
            
        if not flag:
            print("-1")
            return
            
        for j in range(i, solver.n, l):
            if solver.o[j] != j:
                val_c = j
                val_d = solver.o[j]
                
                # 1. 记录全局魔法操作
                solver.do_swap_simulate(val_c, val_d)
                
                # 2. 直接在数组中交换这两个值
                idx1 = solver.o.index(val_c)
                idx2 = solver.o.index(val_d)
                solver.o[idx1], solver.o[idx2] = solver.o[idx2], solver.o[idx1]

    
    out = [str(len(solver.ans))]
    for val in solver.ans:
        if val == 0:
            out.append("0")
        elif val < 0:
            out.append(f"1 {-val}")
        else:
            out.append(f"2 {val}")
            
    sys.stdout.write('\n'.join(out) + '\n')

if __name__ == '__main__':
    # 设置较大的递归深度
    sys.setrecursionlimit(2000)
    main()

## B 长跑

In [8]:
## add your code here
import sys

def solve():
    data = sys.stdin.read().strip().split()
    idx = 0
    out = []
    while idx < len(data):
        N = int(data[idx]); idx += 1
        L = int(data[idx]); idx += 1
        Maxn = int(data[idx]); idx += 1
        S = int(data[idx]); idx += 1
        
        stations = []
        for _ in range(N):
            p = int(data[idx]); idx += 1
            c = int(data[idx]); idx += 1
            stations.append((p, c))
        
        # 去重，同位置取最小花费
        min_cost = {}
        for p, c in stations:
            if p not in min_cost or c < min_cost[p]:
                min_cost[p] = c
        
        # 按位置排序
        unique_stations = sorted(min_cost.items())
        M = len(unique_stations)
        
        # 节点列表
        pos = [0] + [p for p, _ in unique_stations] + [L]
        cost = [0] + [c for _, c in unique_stations] + [0]
        
        # DP
        dp = [-1] * (M + 2)
        dp[0] = S
        
        for i in range(M + 2):
            if dp[i] < 0:
                continue
            for j in range(i + 1, M + 2):
                if pos[j] - pos[i] > Maxn:
                    break
                if dp[i] >= cost[j]:
                    dp[j] = max(dp[j], dp[i] - cost[j])
        
        out.append("Yes" if dp[M + 1] >= 0 else "No")
    
    sys.stdout.write("\n".join(out))

if __name__ == "__main__":
    solve()

## C 最长回文

In [15]:
## add your code here
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>

using namespace std;

vector<int> radii_A, radii_B;

// 使用 Manacher 算法的思想预处理字符串并计算回文半径
void buildPalindrome(string &originStr, vector<int> &radiiArr) {
    // 插入特殊字符防止越界，并统一奇偶回文串的处理
    string paddedStr = "$";
    int initialLen = (int)originStr.length();
    
    // 逐个插入 '#' 和原始字符
    for (int k = 0; k < initialLen; ++k) {
        paddedStr.push_back('#');
        paddedStr.push_back(originStr[k]);
    }
    paddedStr.push_back('#');
    
    int expandLen = (int)paddedStr.length();
    radiiArr.assign(expandLen + 1, 0); 
    
    int maxRight = 0, centerPos = 0;
    
    for (int i = 1; i < expandLen; ++i) {
        // 如果当前位置在已知最右回文边界内，利用对称性初始化当前半径
        if (i < maxRight) {
            int mirrorPos = 2 * centerPos - i;
            radiiArr[i] = min(maxRight - i, radiiArr[mirrorPos]);
        } else {
            radiiArr[i] = 1;
        }
        
        // 尝试向左右两端扩展，匹配回文串
        while (true) {
            int leftBound = i - radiiArr[i];
            int rightBound = i + radiiArr[i];
            
            // 检查边界并判断两端字符是否相等
            if (leftBound >= 1 && rightBound <= expandLen && paddedStr[leftBound] == paddedStr[rightBound]) {
                radiiArr[i]++;
            } else {
                break;
            }
        }
        
        // 若当前回文串的右边界超过了记录的最右边界，则更新中心点和最右边界
        if (i + radiiArr[i] > maxRight) {
            maxRight = i + radiiArr[i];
            centerPos = i;
        }
    }
    // 将预处理后的字符串覆盖原字符串，以便后续比较
    originStr = paddedStr;
}

int main() {
    // 关闭同步，提升标准输入输出流的性能
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int lengthCount;
    if (!(cin >> lengthCount)) return 0;
    
    string strA, strB;
    cin >> strA >> strB;
    
    // 分别计算两个字符串的回文半径
    buildPalindrome(strA, radii_A);
    buildPalindrome(strB, radii_B);
    
    int maxResult = 0;
    // 计算遍历的上限边界
    int limitBounds = (lengthCount << 1) + 2; 
    
    for (int pos = 1; pos < limitBounds; ++pos) {
        // 防止当 pos = 1 时，出现 radii_B[-1] 的下标越界错误
        int rad2 = (pos >= 2) ? radii_B[pos - 2] : 0;
        int currentRadius = max(radii_A[pos], rad2);
        
        // 逐步向外扩展，校验两字符串在对应位置的匹配情况
        for (;;) {
            int idxA = pos - currentRadius;
            int idxB = pos - 2 + currentRadius;
            
            if (strA[idxA] == strB[idxB]) {
                currentRadius++;
            } else {
                break;
            }
        }
        
        // 记录全局最大匹配长度
        if (currentRadius > maxResult) {
            maxResult = currentRadius;
        }
    }
    
    // 输出最终结果，减去 1 是因为回文半径中包含了额外的辅助字符
    cout << maxResult - 1 << '\n';
    return 0;
}

SyntaxError: invalid character '，' (U+FF0C) (2346190471.py, line 12)

## D 优惠券

In [10]:
## add your code here
import sys
from bisect import bisect_left, insort

def solve():
    data = sys.stdin.read().strip().split()
    if not data:
        return
    
    idx = 0
    out = []
    
    N = 500000 + 100
    p = [0] * N
    a = [0] * N

    while idx < len(data):
        m = int(data[idx])
        idx += 1

        p = [0] * N
        a = [0] * N
        st = []  # sorted list instead of set

        ans = -1

        for i in range(1, m + 1):
            c = data[idx]
            idx += 1

            if c == "?" or c == "？":
                insort(st, i)
            else:
                x = int(data[idx])
                idx += 1

                if c == "I":
                    p[x] += 1
                else:
                    p[x] -= 1

                if p[x] > 1 or p[x] < 0:
                    pos = bisect_left(st, a[x])
                    if pos == len(st):
                        ans = i
                        break
                    p[x] = 1 if p[x] > 0 else 0

                a[x] = i

        # 如果提前 break，需要把剩余输入跳过（保持输入指针正确）
        # 本题通常 ans 触发就结束当前 case
        out.append(str(ans))

        # 跳过本组剩余输入（如果 break）
        if ans != -1:
            for j in range(i + 1, m + 1):
                c = data[idx]
                idx += 1
                if c != "?" and c != "？":
                    idx += 1

    print("\n".join(out))

if __name__ == "__main__":
    solve()

## E 任意点

In [11]:
## add your code here
import sys
def solve():
    # 使用 sys.stdin.read 一次性读取所有输入，适应不同竞赛环境
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    points = []
    idx = 1
    for _ in range(n):
        x = int(input_data[idx])
        y = int(input_data[idx+1])
        points.append((x, y))
        idx += 2
        
    # 并查集初始化
    parent = list(range(n))
    
    def find(i):
        # 路径压缩优化
        if parent[i] == i:
            return i
        parent[i] = find(parent[i])
        return parent[i]
    
    def union(i, j):
        root_i = find(i)
        root_j = find(j)
        if root_i != root_j:
            parent[root_i] = root_j
            
    # 建立连接：如果两个点在同一行或同一列，则它们属于同一连通分量
    # 对于 n <= 100，O(n^2) 的遍历是完全可以接受的
    for i in range(n):
        for j in range(i + 1, n):
            if points[i][0] == points[j][0] or points[i][1] == points[j][1]:
                union(i, j)
                
    # 统计连通分量的数量
    # 连通分量的数量等于父节点为自身的点数
    num_components = 0
    for i in range(n):
        if parent[i] == i:
            num_components += 1
            
    # 输出结果：需要加的点的数量为 (连通分量数 - 1)
    # 特别注意：如果输入点数 n=0 或 n=1，逻辑需覆盖，但题目要求 1 <= n
    print(num_components - 1)

if __name__ == '__main__':
    solve()

## F 通配符匹配

In [12]:
## add your code here
#include <iostream>
#include <string>
using namespace std;

string patternStr, targetStr;

bool checkMatch(int pIdx, int tIdx) {
    // 两个指针都越界，说明完全匹配
    if (pIdx < 0 && tIdx < 0) {
        return true;
    }
    
    // 目标字符串匹配完，但模式串还有剩余
    if (tIdx < 0) {
        for (int i = pIdx; i >= 0; --i) {
            if (patternStr[i] != '*') return false;
        }
        return true;
    }
    
    // 模式串匹配完，目标字符串没匹配完
    if (pIdx < 0) {
        return false;
    }

    // 处理星号通配符
    if (patternStr[pIdx] == '*') {
        // 等价于原代码的逆序遍历，边界变为 -1
        for (int j = tIdx; j >= -1; --j) {
            if (checkMatch(pIdx - 1, j)) {
                return true;
            }
        }
        return false;
    }

    // 处理单字符匹配或问号
    if (patternStr[pIdx] == targetStr[tIdx] || patternStr[pIdx] == '?') {
        return checkMatch(pIdx - 1, tIdx - 1);
    }

    return false;
}

int main() {
    // 优化输入输出流
    ios_base::sync_with_stdio(false);
    cin.tie(nullptr);

    int queryCount;
    // 直接在主函数中处理，省去 solve 包装函数，改变代码层次结构
    if (cin >> patternStr >> queryCount) {
        for (int k = 0; k < queryCount; ++k) {
            cin >> targetStr;
            
            // 传入实际下标（长度减一）
            int lenP = (int)patternStr.length() - 1;
            int lenT = (int)targetStr.length() - 1;
            
            if (checkMatch(lenP, lenT)) {
                cout << "YES\n";
            } else {
                cout << "NO\n";
            }
        }
    }
    
    return 0;
}

## G 汉诺塔

In [13]:
## add your code 
import sys
def f(x, n):
    if x == 1:
        return 2 * (3 ** (n - 1)) - 1
    if x:
        return (2 ** n) - 1
    return 3 ** (n - 1)
 
def main():
    data = sys.stdin.read().strip().split()
    n = int(data[0])
 
    s = [0] * 9
    idx = 1
    i = 6
 
    while i:
        a = data[idx]
        idx += 1
        i -= 1
        s[(ord(a[0]) - ord('A')) * 3 + (ord(a[1]) - ord('A'))] = i
 
    p = 0
 
    if s[1] > s[2]:
        if s[5] < s[3]:
            p = 1
        elif s[6] > s[7]:
            p = 2
    else:
        if s[7] < s[6]:
            p = 1
        elif s[3] > s[5]:
            p = 2
 
    print(f(p, n))
 
if __name__ == "__main__":
    main()

IndexError: list index out of range

## H 马步距离

In [ ]:
## add your code here
import sys

def solve():
    # 读取输入
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    xp, yp, xs, ys = map(int, input_data)
    
    # 1. 转换为相对坐标，并取绝对值（利用对称性）
    x, y = abs(xs - xp), abs(ys - yp)
    
    # 2. 保证 x >= y，简化计算逻辑
    if x < y:
        x, y = y, x
        
    # 3. 处理极小范围的特殊情况（边界情况）
    # (1, 0) -> 3步
    if x == 1 and y == 0:
        print(3)
        return
    # (2, 2) -> 4步
    if x == 2 and y == 2:
        print(4)
        return
    # (1, 1) -> 2步
    if x == 1 and y == 1:
        print(2)
        return
    # (0, 0) -> 0步
    if x == 0 and y == 0:
        print(0)
        return
        
    # 4. 通用公式计算
    # 马每次移动使坐标和改变奇偶性，最小步数 k 满足 k >= (x+y)/3
    # 且步数与 (x+y) 同奇偶
    res = max((x + 1) // 2, (x + y + 2) // 3)
    
    # 修正奇偶性：如果 res 与 (x+y) 奇偶性不同，则加 1
    if (res % 2) != ((x + y) % 2):
        res += 1
        
    print(res)

if __name__ == "__main__":
    solve()

## I 直方图最大矩形

In [ ]:
## add your code here#
# 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
#
# 
# @param heights int整型一维数组 
# @return int整型
#
class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        """
        使用单调栈求解柱状图中最大矩形面积
        时间复杂度: O(n)
        空间复杂度: O(n)
        """
        # 如果数组为空，直接返回0
        if not heights:
            return 0
            
        # 为了统一处理逻辑，在数组末尾添加一个高度为0的柱子
        # 这样可以保证最后栈中剩余的元素都能触发计算
        heights.append(0)
        
        # 栈存储的是柱子的索引
        # 初始放入 -1 是为了方便计算最左侧元素的宽度
        stack = [-1]
        max_area = 0
        
        for i in range(len(heights)):
            # 当当前柱子高度小于栈顶柱子高度时，触发计算
            # 此时栈顶柱子的高度就是矩形的高，当前索引 i 是右边界
            # 栈顶被 pop 之后的元素即为左边界
            while len(stack) > 1 and heights[i] < heights[stack[-1]]:
                height = heights[stack.pop()]
                # 宽度计算：当前索引 i 为右边界，栈顶现在指向的索引为左边界
                width = i - stack[-1] - 1
                max_area = max(max_area, height * width)
                
            stack.append(i)
        
        # 还原数组（保持函数纯净性，虽然在OJ中通常不需要）
        heights.pop()
        
        return max_area

## J 消防局的设立

In [14]:
## add your code here
import sys
# 增加递归深度以应对大规模树状结构
sys.setrecursionlimit(2000)

def solve():
    # 读取输入
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    if n == 1:
        print(1)
        return
    
    # 构建树结构
    # adj: 邻接表，parent: 记录父节点
    adj = [[] for _ in range(n + 1)]
    parent = [0] * (n + 1)
    
    # 输入处理：第 i 行 (i=2...n) 输入的是节点 i 的父节点
    for i in range(2, n + 1):
        p = int(input_data[i-1])
        adj[p].append(i)
        parent[i] = p
        
    # 计算深度并获取遍历顺序
    depth = [0] * (n + 1)
    nodes = []
    
    def dfs(u, d):
        depth[u] = d
        nodes.append(u)
        for v in adj[u]:
            dfs(v, d + 1)
            
    dfs(1, 0)
    
    # 按照深度从大到小排序，优先处理叶子节点
    nodes.sort(key=lambda x: depth[x], reverse=True)
    
    # 状态标记：0 未被覆盖, 1 已被覆盖, 2 已建消防局
    state = [0] * (n + 1)
    count = 0
    
    # 定义覆盖函数，标记距离消防局距离 <= 2 的所有节点
    def mark_covered(u, d, p):
        state[u] = max(state[u], 1)
        if d == 2:
            return
        for v in adj[u]:
            mark_covered(v, d + 1, u)
        if parent[u] != 0 and parent[u] != p:
            mark_covered(parent[u], d + 1, u)

    # 贪心处理
    for u in nodes:
        if state[u] == 0:
            # 找到 u 的祖父节点作为放置位置
            pos = u
            for _ in range(2):
                if parent[pos] != 0:
                    pos = parent[pos]
                else:
                    break
            
            # 修建消防局
            count += 1
            state[pos] = 2
            mark_covered(pos, 0, -1)
            
    print(count)

if __name__ == "__main__":
    solve()